# 🎓 AI Tutor
An AI-powered tutor that converts passive technical content into active learning workflows.
**Run cells 1–10 top-to-bottom once**, then interact via **Cell 11** (the tabbed UI).

## Cell 1 — Setup & Dependencies
Installs packages, imports modules, configures the OpenAI client and data directory.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openai', 'graphviz', '-q'], check=False)

import os, json, uuid, random, io
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import Optional

import graphviz
from openai import OpenAI

# ── API Key ───────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    IS_COLAB = True
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
    IS_COLAB = False

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not set. Add it to Colab Secrets or set the env var.')

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Data directory ────────────────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = Path('/content/drive/MyDrive/ai_tutor/data')
else:
    DATA_DIR = Path('./data')


DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature flags ─────────────────────────────────────────────────────────────
ENABLE_WEB_SUGGESTIONS = False  # Set False to skip the post-ingest web search step

print(f'✓ Setup complete | Data dir: {DATA_DIR} | Colab: {IS_COLAB}')


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


✓ Setup complete | Data dir: data | Colab: False


## Cell 2 — Data Models & Persistence (`Store`)
Defines `Content`, `Topic`, and `SessionRecord` dataclasses plus the `Store` class that owns all JSON I/O. Every other class reads and writes through `Store`.

In [2]:
@dataclass
class Content:
    id: str
    title: str
    body: str
    topic_id: str
    source_file: str
    created_at: str
    vector_file_id: str = ''


@dataclass
class Topic:
    id: str
    name: str
    parent_id: Optional[str]
    children_topic_ids: list = field(default_factory=list)
    content_ids: list = field(default_factory=list)


@dataclass
class SessionRecord:
    date: str
    content_id: str
    question_type: str   # 'mcq' | 'open'
    correct: bool
    score: float


class Store:
    """Single source of truth for all JSON persistence."""

    TREE_FILE     = 'content_tree.json'
    VS_FILE       = 'vector_store.json'
    PROGRESS_FILE = 'user_progress.json'

    def __init__(self, data_dir: Path):
        self.data_dir = data_dir
        self.topics: dict = {}
        self.contents: dict = {}
        self.vector_store_id: str = ''
        self.sessions: list = []
        self.weak_content_ids: list = []
        self.load()

    # ── I/O ──────────────────────────────────────────────────────────────────

    def load(self):
        self._load_tree()
        self._load_vs()
        self._load_progress()

    def save(self):
        self._save_tree()
        self._save_vs()
        self._save_progress()

    def _load_tree(self):
        path = self.data_dir / self.TREE_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.topics   = {k: Topic(**v)   for k, v in data.get('topics', {}).items()}
            self.contents = {k: Content(**v) for k, v in data.get('contents', {}).items()}
        else:
            root = Topic(id=str(uuid.uuid4()), name='Root', parent_id=None)
            self.topics[root.id] = root
            self._save_tree()

    def _save_tree(self):
        (self.data_dir / self.TREE_FILE).write_text(json.dumps(
            {'topics': {k: asdict(v) for k, v in self.topics.items()},
             'contents': {k: asdict(v) for k, v in self.contents.items()}},
            indent=2
        ))

    def _load_vs(self):
        path = self.data_dir / self.VS_FILE
        if path.exists():
            self.vector_store_id = json.loads(path.read_text()).get('vector_store_id', '')

    def _save_vs(self):
        (self.data_dir / self.VS_FILE).write_text(
            json.dumps({'vector_store_id': self.vector_store_id}, indent=2)
        )

    def _load_progress(self):
        path = self.data_dir / self.PROGRESS_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.sessions          = [SessionRecord(**s) for s in data.get('sessions', [])]
            self.weak_content_ids  = data.get('weak_content_ids', [])
        else:
            self._save_progress()

    def _save_progress(self):
        (self.data_dir / self.PROGRESS_FILE).write_text(json.dumps(
            {'sessions': [asdict(s) for s in self.sessions],
             'weak_content_ids': self.weak_content_ids},
            indent=2
        ))

    # ── Accessors ─────────────────────────────────────────────────────────────

    def get_topic(self, id: str) -> Optional[Topic]:
        return self.topics.get(id)

    def get_content(self, id: str) -> Optional[Content]:
        return self.contents.get(id)

    def add_topic(self, topic: Topic):
        self.topics[topic.id] = topic
        self._save_tree()

    def add_content(self, content: Content):
        self.contents[content.id] = content
        self._save_tree()

    def delete_content(self, id: str):
        content = self.contents.pop(id, None)
        if content:
            topic = self.topics.get(content.topic_id)
            if topic and id in topic.content_ids:
                topic.content_ids.remove(id)
            self._save_tree()
        return content

    def all_content(self) -> list:
        return list(self.contents.values())

    def all_topics(self) -> list:
        return list(self.topics.values())

    def root_topic(self) -> Optional[Topic]:
        return next((t for t in self.topics.values() if t.parent_id is None), None)


store = Store(DATA_DIR)
print(f'✓ Store loaded | Topics: {len(store.topics)} | Content items: {len(store.contents)}')

✓ Store loaded | Topics: 22 | Content items: 76


## Cell 3 — Content Management (`ContentManager`)
Wraps `Store` to provide topic/content tree CRUD, navigation, and graphviz tree visualisation.
`render_tree()` returns a `graphviz.Digraph`; call `display(cm.render_tree())` to show it inline.

In [3]:
class ContentManager:
    """Manages navigation and editing of the topic/content tree."""

    def __init__(self, store: Store):
        self.store = store

    def create_topic(self, name: str, parent_id: str = None) -> Topic:
        if parent_id is None:
            root = self.store.root_topic()
            parent_id = root.id if root else None
        topic = Topic(id=str(uuid.uuid4()), name=name, parent_id=parent_id)
        self.store.add_topic(topic)
        if parent_id:
            parent = self.store.get_topic(parent_id)
            if parent and topic.id not in parent.children_topic_ids:
                parent.children_topic_ids.append(topic.id)
                self.store._save_tree()
        return topic

    def add_content_to_topic(self, content: Content, topic_id: str):
        self.store.add_content(content)
        topic = self.store.get_topic(topic_id)
        if topic and content.id not in topic.content_ids:
            topic.content_ids.append(content.id)
            self.store._save_tree()

    def list_topics(self, parent_id: str = None, indent: int = 0) -> str:
        if parent_id is None:
            root = self.store.root_topic()
            if not root:
                return '(empty)'
            parent_id = root.id
        topic = self.store.get_topic(parent_id)
        if not topic:
            return ''
        lines = ['  ' * indent + f'📁 {topic.name}']
        for cid in topic.content_ids:
            c = self.store.get_content(cid)
            if c:
                lines.append('  ' * (indent + 1) + f'📄 {c.title}')
        for child_id in topic.children_topic_ids:
            lines.append(self.list_topics(child_id, indent + 1))
        return '\n'.join(lines)

    def view_content(self, content_id: str) -> Optional[Content]:
        return self.store.get_content(content_id)

    def delete_content(self, content_id: str) -> str:
        """Returns vector_file_id for upstream cleanup."""
        content = self.store.delete_content(content_id)
        return content.vector_file_id if content else ''

    def move_content(self, content_id: str, new_topic_id: str):
        content = self.store.get_content(content_id)
        if not content:
            return
        old_topic = self.store.get_topic(content.topic_id)
        if old_topic and content_id in old_topic.content_ids:
            old_topic.content_ids.remove(content_id)
        content.topic_id = new_topic_id
        new_topic = self.store.get_topic(new_topic_id)
        if new_topic and content_id not in new_topic.content_ids:
            new_topic.content_ids.append(content_id)
        self.store._save_tree()

    def render_tree(self):
        """Returns a graphviz.Digraph — call display(cm.render_tree()) to render inline."""
        dot = graphviz.Digraph(graph_attr={'rankdir': 'TB', 'splines': 'ortho'})
        dot.attr('node', fontname='Helvetica')
        for topic in self.store.all_topics():
            dot.node(topic.id, label=f'📁 {topic.name}', shape='folder',
                     style='filled', fillcolor='#dce8f5')
            if topic.parent_id:
                dot.edge(topic.parent_id, topic.id)
            for cid in topic.content_ids:
                content = self.store.get_content(cid)
                if content:
                    dot.node(cid, label=f'📄 {content.title}', shape='note',
                             style='filled', fillcolor='#f5f0dc')
                    dot.edge(topic.id, cid)
        return dot


content_manager = ContentManager(store)
print('✓ ContentManager ready')

✓ ContentManager ready


## Cell 4 — ReAct Agent (`Agent`)
Reusable ReAct engine. Subclass it to build agents that follow the
**Think → Act → Observe** loop using OpenAI function calling.

- `_fn_tool` — builds a strict function-tool definition
- `_think` — one LLM call; returns the single function-call decision
- `_act` — executes a tool call; returns the observation string
- `_react_loop` — orchestrates the loop until the finalization tool is called

In [4]:
class Agent:
    """Reusable ReAct engine: Think → Act → Observe via OpenAI function calling."""

    MAX_REACT_CYCLES = 10

    def __init__(self, openai_client):
        self.client = openai_client

    # ── Tool definition ───────────────────────────────────────────────────────

    @staticmethod
    def _fn_tool(name: str, description: str, properties: dict, required: list = None) -> dict:
        """Build a function tool definition following the OpenAI function-calling spec."""
        return {
            'type': 'function',
            'name': name,
            'description': description,
            'parameters': {
                'type': 'object',
                'properties': properties,
                'required': required or [],
                'additionalProperties': False,
            },
            'strict': True,
        }

    # ── ReAct phases ──────────────────────────────────────────────────────────

    def _think(self, next_input: list, tools: list, prev_id: str | None):
        """
        Think phase: call the model and return its single function-call decision.

        Returns (response_id, calls) where each call is {name, call_id, args}.
        parallel_tool_calls=False ensures one call per response so each
        observation is visible before the next action is chosen.
        """
        kwargs = dict(model='gpt-4o', input=next_input, tools=tools, parallel_tool_calls=False)
        if prev_id:
            kwargs['previous_response_id'] = prev_id
        response = self.client.responses.create(**kwargs)

        response_id = getattr(response, 'id', None)
        calls = []
        for item in getattr(response, 'output', []) or []:
            if getattr(item, 'type', None) != 'function_call':
                continue
            raw = getattr(item, 'arguments', '{}') or '{}'
            try:
                args = json.loads(raw)
            except Exception:
                args = {}
            calls.append({
                'name':    getattr(item, 'name', ''),
                'call_id': getattr(item, 'call_id', ''),
                'args':    args,
            })
        return response_id, calls

    def _act(self, call: dict, execute_fn) -> str:
        """Act phase: execute one tool call and return the observation string."""
        return execute_fn(call['name'], call['args'])

    # ── Loop orchestrator ─────────────────────────────────────────────────────

    def _react_loop(
        self,
        initial_input: list,
        tools: list,
        final_tool_name: str,
        execute_fn,
        step_name: str,
        log_fn=None,
        show_reasoning: bool = True,
    ) -> dict:
        """
        Orchestrate Think → Act → Observe until `final_tool_name` is called.

        Each cycle: _think() returns one decision; if it is a regular tool,
        _act() executes it and the observation feeds the next cycle; if it is
        `final_tool_name`, its arguments are returned as the step result.
        Conversation state persists across cycles via previous_response_id.

        Args:
            initial_input:    First messages sent to the model.
            tools:            Tool definitions available this step.
            final_tool_name:  Calling this tool ends the loop.
            execute_fn:       fn(tool_name, args) -> observation str.
            step_name:        Label used in log lines.
            log_fn:           Callable for log output; defaults to print.
            show_reasoning:   Whether to emit [Think] lines.
        """
        _log       = log_fn or print
        prev_id    = None
        next_input = initial_input

        for cycle in range(1, self.MAX_REACT_CYCLES + 1):
            try:
                prev_id, calls = self._think(next_input, tools, prev_id)
            except Exception as exc:
                _log(f'    [Think   {cycle}] ✗ API error: {exc}')
                break

            if not calls:
                _log(f'    [Think   {cycle}] No tool selected — prompting continuation.')
                next_input = [{
                    'role': 'user',
                    'content': (
                        f'No tool was called. Continue reasoning, use a tool, '
                        f'or call `{final_tool_name}` when satisfied.'
                    ),
                }]
                continue

            tool_outputs = []
            final_result = None

            for call in calls:
                name    = call['name']
                call_id = call['call_id']
                args    = call['args']
                display = {k: v for k, v in args.items() if k != 'thought'}

                thought = (args.get('thought') or '').strip()
                if thought and show_reasoning:
                    _log(f'    [Think   {cycle}] 💭 {thought}')

                _log(f'    [Act     {cycle}] ⚙️  {name}({json.dumps(display, ensure_ascii=True)})')

                if name == final_tool_name:
                    final_result = args
                    if call_id:
                        tool_outputs.append({
                            'type':    'function_call_output',
                            'call_id': call_id,
                            'output':  'Submission accepted.',
                        })
                else:
                    observation = self._act(call, execute_fn)
                    _log(f'    [Observe {cycle}] 👀 {str(observation)}')
                    if call_id:
                        tool_outputs.append({
                            'type':    'function_call_output',
                            'call_id': call_id,
                            'output':  str(observation),
                        })

            if final_result is not None:
                _log(f'    [Done] {step_name} complete at cycle {cycle}')
                return final_result

            next_input = tool_outputs

        _log(f'    [Done] {step_name}: max cycles reached without completion.')
        return {}


print('✓ Agent class defined')


✓ Agent class defined


## Cell 5 — Content Ingestion (`Ingester`)
Subclasses `Agent` and implements the two-step ingestion ReAct workflow.

**Step 1 — Outline** (`submit_outline`): agent reads the document, inspects the
topic tree, creates missing topics top-down, and submits `[{topic_id, topic_name}]`.

**Step 2 — Categorize** (`submit_categorization`): agent assigns every concept
to one of the outline topic IDs and submits `[{title, body, topic_id}]`.

Pipeline: **upload → ReAct outline → ReAct categorize → store → embed → web-search → cleanup**

In [5]:
class Ingester(Agent):
    """ReAct agent for content ingestion: upload → outline → categorize → store → embed."""

    SUPPORTED_EXTS = {'.pdf', '.png', '.jpg', '.jpeg', '.webp', '.gif'}
    def __init__(self, openai_client, content_manager, vector_store, web_searcher,
                 suggest_similar: bool = True, show_reasoning: bool = True):
        super().__init__(openai_client)
        self.cm              = content_manager
        self.vs              = vector_store
        self.ws              = web_searcher
        self.suggest_similar = suggest_similar
        self.show_reasoning  = show_reasoning
        self._active_log_fn  = print

    # ── Public entry point ────────────────────────────────────────────────────

    def ingest(self, file_bytes: bytes, filename: str, log_fn=None) -> list:
        """Full pipeline: upload → ReAct outline → ReAct categorize → store → embed → suggest → cleanup."""
        ext = Path(filename).suffix.lower()
        if ext not in self.SUPPORTED_EXTS:
            raise ValueError(f'Unsupported type "{ext}". Supported: {self.SUPPORTED_EXTS}')

        prev = self._active_log_fn
        self._active_log_fn = log_fn or print
        try:
            self._log(f'⬆  Uploading "{filename}"…')
            file_id = self._upload_file(file_bytes, filename)

            self._log('🧠 ReAct — Step 1/2: Draft outline…')
            outline_topics = self._run_outline_step(file_id)
            self._log(f'  ✓ {len(outline_topics)} topic(s) in outline')

            self._log('🧠 ReAct — Step 2/2: Categorize content…')
            concepts = self._run_categorization_step(
                file_id=file_id,
                outline_topics=outline_topics,
            )
            self._log(f'  ✓ {len(concepts)} concept(s) across {len(outline_topics)} topic(s)')

            self._log('💾 Storing and embedding concepts…')
            root_id = self.cm.store.root_topic().id
            created = []
            for concept in concepts:
                resolved_topic = (concept.get('topic_id') or '').strip()
                if not resolved_topic or not self.cm.store.get_topic(resolved_topic):
                    resolved_topic = root_id
                c = Content(
                    id=str(uuid.uuid4()),
                    title=concept.get('title', 'Untitled'),
                    body=concept.get('body', ''),
                    topic_id=resolved_topic,
                    source_file=filename,
                    created_at=datetime.utcnow().isoformat(),
                )
                c.vector_file_id = self.vs.add_content(c)
                self.cm.add_content_to_topic(c, resolved_topic)
                created.append(c)

            if self.suggest_similar:
                self._log('🔍 Searching for related learning resources…')
                for c in created:
                    suggestions = self.ws.suggest(c.title)
                    if suggestions:
                        self._log(f'\n  📌 Resources for "{c.title}":')
                        for s in suggestions[:3]:
                            self._log(f'     • {s.get("title", "")}')
                            self._log(f'       {s.get("url", "")}')

            self._log('\n🗑  Cleaning up upload…')
            self._delete_upload(file_id)
            self._log(f'\n✅ Ingestion complete — {len(created)} concept(s) added.')
            return created
        finally:
            self._active_log_fn = prev

    def _outline_tools(self) -> list:
        return [
            self._fn_tool(
                'observe_tree',
                'Inspect the full content-tree to understand existing topics and their IDs before deciding placement.',
                {
                    'thought': {
                        'type': 'string',
                        'description': 'Your reasoning for this action.',
                    },
                },
                ['thought'],
            ),
            self._fn_tool(
                'add_topic',
                'Create a new topic under a chosen parent. Call observe_tree first to pick the right parent_topic_id.',
                {
                    'thought': {
                        'type': 'string',
                        'description': 'Your reasoning for this action.',
                    },
                    'topic_name': {
                        'type': 'string',
                        'description': 'Name for the new topic.',
                    },
                    'parent_topic_id': {
                        'type': 'string',
                        'description': 'ID of the parent topic. Must be a valid ID from observe_tree.',
                    },
                },
                ['thought', 'topic_name', 'parent_topic_id'],
            ),
            self._fn_tool(
                'submit_outline',
                (
                    'Submit the final list of topics for this document. '
                    'Each entry must have a real topic_id that already exists in the tree '
                    '(either reused or just created with add_topic). '
                    'Terminates the ReAct loop for this step.'
                ),
                {
                    'thought': {
                        'type': 'string',
                        'description': 'Reasoning for the topic structure chosen.',
                    },
                    'topics': {
                        'type': 'array',
                        'description': 'Topics covering this document — both reused and newly created.',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'topic_id':   {'type': 'string', 'description': 'Real ID in the content tree.'},
                                'topic_name': {'type': 'string', 'description': 'Display name.'},
                            },
                            'required': ['topic_id', 'topic_name'],
                            'additionalProperties': False,
                        },
                    },
                },
                ['thought', 'topics'],
            ),
        ]

    def _categorize_tools(self, outline_topics: list) -> list:
        topic_ref = ', '.join(
            f'{t["topic_name"]} (id={t["topic_id"]})'
            for t in outline_topics
        )
        return [
            self._fn_tool(
                'submit_categorization',
                (
                    'Submit the extracted concepts, each assigned to one of the outline topics. '
                    'Terminates the ReAct loop for this step.'
                ),
                {
                    'thought': {
                        'type': 'string',
                        'description': 'Final reasoning summary.',
                    },
                    'concepts': {
                        'type': 'array',
                        'description': 'All learning concepts extracted from the document.',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'title': {
                                    'type': 'string',
                                    'description': 'Short concept title.',
                                },
                                'body': {
                                    'type': 'string',
                                    'description': 'Detailed explanation suitable for question generation.',
                                },
                                'topic_id': {
                                    'type': 'string',
                                    'description': (
                                        f'Must be one of the outline topic IDs: {topic_ref}.'
                                    ),
                                },
                            },
                            'required': ['title', 'body', 'topic_id'],
                            'additionalProperties': False,
                        },
                    },
                },
                ['thought', 'concepts'],
            ),
        ]
    # ── Step 1: Draft outline ─────────────────────────────────────────────────

    def _run_outline_step(self, file_id: str) -> list:
        """
        Outline step: agent reads the document, inspects the tree, creates any needed
        topics via add_topic, then submits a concrete list of {topic_id, topic_name}.
        Returns that list — all IDs are already live in the content tree.
        """
        tools = self._outline_tools()
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a ReAct ingestion agent.\n'
                        'Goal: place this document into a well-grouped, nested topic hierarchy and '
                        'return the topics that concepts will be directly assigned to.\n\n'
                        'Tree structure rules:\n'
                        '  - Root-level topics represent broad Computer Science domains '
                        '    (e.g. Programming Languages, Databases, Algorithms & Data Structures, '
                        '    Operating Systems, Networks, Machine Learning, Software Engineering).\n'
                        '  - Group related topics under a shared parent whenever possible — '
                        '    prefer nesting over a flat list of siblings under Root.\n'
                        '  - Depth is flexible: two levels may be enough for a broad topic, '
                        '    three or more for a specific one. Let the content dictate the depth.\n'
                        '  - Topics submitted in submit_outline are the ones concepts attach to; '
                        '    pure grouping parents do not need to be included.\n\n'
                        'How to work:\n'
                        '  - One tool call at a time. Always read the observation before deciding the next action.\n'
                        '  - Use the `thought` field to reason about what the observation tells you \n'
                        '    and what you will do next — especially when a tool returns an error.\n'
                        '  - If a tool call fails, diagnose the error from the observation and fix it \n'
                        '    (e.g. call `observe_tree` to get valid IDs) before retrying.\n'
                        '  - Start with `observe_tree`, then create missing topics top-down \n'
                        '    (broader groups first, using the real ID returned by each `add_topic`).\n'
                        '  - Call `submit_outline` only when all needed topics exist in the tree.\n\n'
                        'Quality criteria:\n'
                        '  - Every topic_id in submit_outline already exists in the tree.\n'
                        '  - Root children are broad CS domains, not document-specific titles.\n'
                        '  - No near-duplicate topics at any level.\n'
                        '  - Topics are specific enough for concept-level categorization.\n'
                    ),
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'input_file', 'file_id': file_id},
                        {'type': 'input_text', 'text': (
                            'Read this document and build its topic hierarchy.\n'
                            'Think before every action (use the `thought` field), and always \n'
                            'react to what the observation tells you before calling the next tool.'
                        )},
                    ],
                },
            ],
            tools=tools,
            final_tool_name='submit_outline',
            execute_fn=lambda name, args: self._execute_tool(name, args),
            step_name='outline',
            log_fn=self._log,
            show_reasoning=self.show_reasoning,
        )
        if self.show_reasoning and result.get('thought'):
            self._log(f'    💭 Final thought: {result["thought"][:200]}')
        return result.get('topics', [])

    # ── Step 2: Categorize content ────────────────────────────────────────────

    def _run_categorization_step(self, file_id: str, outline_topics: list) -> list:
        """
        Categorize step: agent assigns every concept in the document to one of the
        topics produced by the outline step. No new topics are created here.
        Returns a flat list of concept dicts with {title, body, topic_id}.
        """
        tools = self._categorize_tools(outline_topics)
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a ReAct ingestion agent.\n'
                        'Goal: read the document and extract every distinct concept, '
                        'assigning each to one of the provided outline topics.\n\n'
                        'Rules:\n'
                        '  - Use only the topic_ids listed in the outline — do not invent new ones.\n'
                        '  - Every concept needs a title, a detailed body, and a valid topic_id.\n'
                        '  - Concept bodies must be detailed enough for question generation.\n'
                        '  - Call `submit_categorization` when all concepts are extracted.\n'
                    ),
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'input_file', 'file_id': file_id},
                        {'type': 'input_text', 'text': (
                            f'Outline topics (use only these IDs):\n'
                            f'{json.dumps(outline_topics, ensure_ascii=True)}\n\n'
                            'Extract all concepts from the document and assign each to one of the '
                            'topic_ids above. Call `submit_categorization` when done.'
                        )},
                    ],
                },
            ],
            tools=tools,
            final_tool_name='submit_categorization',
            execute_fn=lambda name, args: self._execute_tool(name, args),
            step_name='categorize',
            log_fn=self._log,
            show_reasoning=self.show_reasoning,
        )
        if self.show_reasoning and result.get('thought'):
            self._log(f'    💭 Final thought: {result["thought"][:200]}')
        return result.get('concepts', [])

    # ── Tool executor ─────────────────────────────────────────────────────────

    def _execute_tool(self, tool_name: str, args: dict) -> str:
        """Dispatch a tool call and return an observation string."""
        tool_name = (tool_name or '').strip().lower()
        args      = args or {}

        if tool_name == 'observe_tree':
            snapshot = self._topic_state()
            return f'{len(snapshot)} topic(s) in tree: {json.dumps(snapshot, ensure_ascii=True)}'

        if tool_name == 'add_topic':
            topic_name = (args.get('topic_name') or '').strip()
            parent_id  = (args.get('parent_topic_id') or '').strip()
            if not topic_name:
                return 'Error: topic_name is required.'
            if not parent_id or not self.cm.store.get_topic(parent_id):
                return (
                    'Error: parent_topic_id is missing or invalid. '
                    'Call observe_tree to find valid topic IDs.'
                )
            existing = self._find_topic_by_name(topic_name)
            if existing:
                return (
                    f'Skipped: topic "{existing.name}" ({existing.id}) already exists — '
                    f'reuse it instead of creating a duplicate.'
                )
            created = self.cm.create_topic(topic_name, parent_id)
            return f'Created: "{created.name}" (id={created.id}) under parent={parent_id}.'

        return f'Unknown tool: {tool_name}'

    # ── Persistence helpers ───────────────────────────────────────────────────

    def _topic_state(self) -> list:
        return [
            {'id': t.id, 'name': t.name, 'parent_id': t.parent_id}
            for t in self.cm.store.all_topics()
        ]

    def _find_topic_by_name(self, name: str):
        key = self._normalize(name)
        for t in self.cm.store.all_topics():
            if self._normalize(t.name) == key:
                return t
        return None

    def _find_topic_by_name_under_parent(self, name: str, parent_id: str):
        key = self._normalize(name)
        for t in self.cm.store.all_topics():
            if t.parent_id == parent_id and self._normalize(t.name) == key:
                return t
        return None

    @staticmethod
    def _normalize(text: str) -> str:
        return (text or '').strip().lower()

    # ── File API helpers ──────────────────────────────────────────────────────

    def _upload_file(self, file_bytes: bytes, filename: str) -> str:
        ext  = Path(filename).suffix.lower()
        mime = 'application/pdf' if ext == '.pdf' else 'image/png'
        resp = self.client.files.create(file=(filename, file_bytes, mime), purpose='assistants')
        return resp.id

    def _delete_upload(self, file_id: str):
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass

    # ── Logging ───────────────────────────────────────────────────────────────

    def _log(self, message: str):
        try:
            self._active_log_fn(message)
        except Exception:
            print(message)

    @staticmethod
    def _parse_json(text: str) -> dict:
        """Strip markdown fences and parse the first JSON object/array found."""
        import re
        text = text.strip()
        # Remove ```json ... ``` or ``` ... ``` fences
        text = re.sub(r'^```[a-zA-Z]*\n?', '', text)
        text = re.sub(r'```$', '', text.strip())
        # Find the outermost { } or [ ]
        for start_ch, end_ch in [('{', '}'), ('[', ']')]:
            s = text.find(start_ch)
            if s == -1:
                continue
            depth = 0
            for e, ch in enumerate(text[s:], start=s):
                if ch == start_ch:
                    depth += 1
                elif ch == end_ch:
                    depth -= 1
                    if depth == 0:
                        return json.loads(text[s:e+1])
        return json.loads(text)


print('✓ Ingester defined (ReAct agent with OpenAI function calling)')


✓ Ingester defined (ReAct agent with OpenAI function calling)


## Cell 6 — Vector Store (`VectorStore`)
Wraps the OpenAI Vector Stores API. Creates (or reuses) a persisted vector store, uploads concept text, and runs semantic search via the `file_search` Responses API tool.

In [6]:
class VectorStore:
    """Wraps the OpenAI Vector Stores API for semantic search over ingested content."""

    STORE_NAME = 'ai_tutor'

    def __init__(self, openai_client, store: Store):
        self.client = openai_client
        self.store  = store

    def get_or_create(self) -> str:
        """Return vector store ID, creating one on first call."""
        if self.store.vector_store_id:
            return self.store.vector_store_id
        vs = self.client.vector_stores.create(name=self.STORE_NAME)
        self.store.vector_store_id = vs.id
        self.store._save_vs()
        print(f'✓ Vector store created: {vs.id}')
        return vs.id

    def add_content(self, content: Content) -> str:
        """Upload concept body as plain text and attach to vector store. Returns file_id."""
        vs_id     = self.get_or_create()
        file_obj  = self.client.files.create(
            file=(f'{content.id}.txt', content.body.encode('utf-8'), 'text/plain'),
            purpose='assistants'
        )
        self.client.vector_stores.files.create(vector_store_id=vs_id, file_id=file_obj.id)
        return file_obj.id

    def delete_content(self, file_id: str):
        """Remove from vector store and Files API (best-effort)."""
        if not file_id:
            return
        vs_id = self.store.vector_store_id
        if vs_id:
            try:
                self.client.vector_stores.files.delete(vector_store_id=vs_id, file_id=file_id)
            except Exception:
                pass
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass

    def search(self, query: str, top_k: int = 5) -> list:
        """Return content_ids of the top-k semantically matching concepts."""
        vs_id = self.get_or_create()
        if not self.store.all_content():
            return []
        try:
            response = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'file_search', 'vector_store_ids': [vs_id]}],
                input=(
                    f'Find the most relevant concepts for: {query}\n\n'
                    'Return ONLY a JSON array of matched concept titles in relevance order. No other text.'
                )
            )
            titles    = Ingester._parse_json(response.output_text)
            title_map = {c.title: c.id for c in self.store.all_content()}
            return [title_map[t] for t in titles if t in title_map][:top_k]
        except Exception:
            return []


vector_store = VectorStore(client, store)
print('✓ VectorStore ready')

✓ VectorStore ready


## Cell 7 — Web Search (`WebSearcher`)
Surfaces related learning resources after ingestion using the OpenAI Responses API `web_search_preview` built-in tool — no extra API key required.

In [7]:
class WebSearcher:
    """Surfaces related resources via OpenAI Responses API web_search_preview."""

    def __init__(self, openai_client):
        self.client = openai_client

    def suggest(self, concept_title: str) -> list:
        """Return up to 5 relevant resources for the given concept title."""
        try:
            response = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'web_search_preview'}],
                input=(
                    f'Find the best 3-5 online learning resources, tutorials, or official docs '
                    f'for someone studying: "{concept_title}".\n'
                    'Return ONLY a JSON array — no markdown, no commentary:\n'
                    '[{"title": "...", "url": "...", "summary": "..."}]'
                )
            )
            return Ingester._parse_json(response.output_text)
        except Exception:
            return []


web_searcher = WebSearcher(client)
print('✓ WebSearcher ready')

✓ WebSearcher ready


## Cell 8 — Question Generation (`QuestionEngine`)
Generates MCQ or open-ended questions using a two-step Think Architecture chain.
Practical **code questions are restricted to Python or SQL** so they can be executed by Code Interpreter.
Also generates optional follow-up questions when the student shows partial understanding.

In [8]:
class QuestionEngine(Agent):
    """ReAct agent that designs a personalised question set for a study concept."""

    MAX_FOLLOWUPS = 3

    # ── Public API ─────────────────────────────────────────────────────────────

    def design_question_set(self, content: Content, log_fn=None) -> list:
        """
        Two-step ReAct pipeline: plan which questions to ask, then design each one.
        Returns a list of question dicts ready for the study presenter.
        """
        _log = log_fn or print

        _log('\n🧠 ReAct — Step 1/2: Planning question set…')
        plan = self._run_plan_step(content, log_fn)
        if not plan:
            _log('  ⚠  No questions planned.')
            return []
        _log(f'  ✓ {len(plan)} question(s) planned:')
        for i, p in enumerate(plan, 1):
            _log(f'     {i}. [{p["question_type"].upper()} / {p["question_style"]}] {p["focus"]}')

        _log('\n🧠 ReAct — Step 2/2: Designing each question…')
        questions = self._run_design_step(content, plan, log_fn)
        _log(f'  ✓ {len(questions)} question(s) ready.')
        return questions

    # ── Step 1: Plan ──────────────────────────────────────────────────────────

    def _run_plan_step(self, content: Content, log_fn=None) -> list:
        """Agent reads the concept and drafts a plan of which questions to ask and why."""
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a question planning agent.\n'
                        'Task: read a learning concept and draft a plan of questions to design.\n\n'
                        'Possible question types:\n'
                        '  - Theory MCQ:     multiple-choice on conceptual understanding\n'
                        '  - Theory Open:    student explains in their own words\n'
                        '  - Practical MCQ:  scenario-based multiple-choice\n'
                        '  - Practical Open: apply the concept; Python/SQL code if it fits\n\n'
                        'Rules:\n'
                        '  - Only plan types that are genuinely valuable for THIS concept.\n'
                        '  - For each planned question, write a clear `focus`: what it tests '
                        'and why it helps the student learn (not memorise).\n'
                        '  - Do not write the actual questions yet — just the plan.\n'
                        '  - Call submit_plan when your plan is complete.'
                    ),
                },
                {
                    'role': 'user',
                    'content': (
                        f'Concept: {content.title}\n\n{content.body}\n\n'
                        'Draft a question plan for this concept.'
                    ),
                },
            ],
            tools=[self._plan_tool()],
            final_tool_name='submit_plan',
            execute_fn=lambda name, args: 'Plan received.',
            step_name='plan',
            log_fn=log_fn,
            show_reasoning=True,
        )
        return result.get('questions', [])

    # ── Step 2: Design ────────────────────────────────────────────────────────

    def _run_design_step(self, content: Content, plan: list, log_fn=None) -> list:
        """Agent takes the plan and designs each question in full detail."""
        state = {'questions': []}

        def execute_fn(name, args):
            if name == 'add_mcq_question':
                q = {k: v for k, v in args.items() if k != 'thought'}
                q['question_type'] = 'mcq'
                q['content_id']    = content.id
                state['questions'].append(q)
                return f'MCQ added ({q["question_style"]}). Total: {len(state["questions"])}'
            if name == 'add_open_question':
                q = {k: v for k, v in args.items() if k != 'thought'}
                q['question_type'] = 'open'
                q['content_id']    = content.id
                state['questions'].append(q)
                lang = q.get('language', 'none')
                tag  = f' [{lang.upper()}]' if lang != 'none' else ''
                return f'Open{tag} added ({q["question_style"]}). Total: {len(state["questions"])}'
            if name == 'skip_question':
                idx = args.get('plan_index', '?')
                return f'Skipped planned question {idx}: {args.get("reason", "")}'
            return f'Unknown tool: {name}'

        plan_text = '\n'.join(
            f'  {i+1}. [{p["question_type"].upper()} / {p["question_style"]}] {p["focus"]}'
            for i, p in enumerate(plan)
        )

        self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a question design agent.\n'
                        'Task: design each question from the plan in full detail.\n\n'
                        'Rules:\n'
                        '  - Work through the plan in order, one question at a time.\n'
                        '  - Use add_mcq_question for MCQ entries, add_open_question for Open entries.\n'
                        '  - If a planned question cannot be designed as intended, call skip_question.\n'
                        '  - Code questions (Practical Open): Python or SQL ONLY.\n'
                        '  - Focus on understanding and application, not memorisation.\n'
                        '  - Call submit_question_set when all planned questions are done.'
                    ),
                },
                {
                    'role': 'user',
                    'content': (
                        f'Concept: {content.title}\n\n{content.body}\n\n'
                        f'Plan to execute:\n{plan_text}\n\n'
                        'Design each planned question now, in order.'
                    ),
                },
            ],
            tools=self._design_tools(),
            final_tool_name='submit_question_set',
            execute_fn=execute_fn,
            step_name='design',
            log_fn=log_fn,
            show_reasoning=True,
        )
        return state['questions']

    def generate_followup(self, content: Content, prior_question: dict,
                          student_answer: str, score: float) -> Optional[dict]:
        """Return a follow-up open-ended question, or None if not warranted."""
        fmt_hint = 'open-ended JSON: {"question":"...","language":"none","model_answer":"...","rubric":"...","test_code":""}'
        response = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'Concept: {content.title}\n{content.body}\n\n'
                f'The student was asked:\n{prior_question.get("question", "")}\n\n'
                f'Their answer: {student_answer}\nScore: {score:.1f}/1.0\n\n'
                'Based on the concept, the question, and the student\'s answer: '
                'is there a meaningful gap in understanding that a follow-up question would help address? '
                'Only ask a follow-up if it genuinely deepens learning — do not ask for the sake of it.\n'
                f'If yes, write one follow-up open-ended question as {fmt_hint}.\n'
                'If no follow-up is needed, output exactly: NO_FOLLOWUP'
            )
        )
        raw = response.output_text.strip()
        if 'NO_FOLLOWUP' in raw:
            return None
        try:
            q = Ingester._parse_json(raw)
            q['content_id']     = content.id
            q['question_type']  = 'open'
            q['question_style'] = prior_question.get('question_style', 'theory')
            return q
        except Exception:
            return None

    # ── Tool definitions ───────────────────────────────────────────────────────

    def _plan_tool(self) -> dict:
        return self._fn_tool(
            'submit_plan',
            'Submit the question plan. Each entry says what type to design and what it should test.',
            {
                'thought': {
                    'type': 'string',
                    'description': 'Overall reasoning: what angles are worth testing for this concept.',
                },
                'questions': {
                    'type': 'array',
                    'description': 'Ordered list of questions to design.',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'question_type':  {'type': 'string', 'enum': ['mcq', 'open']},
                            'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                            'focus': {
                                'type': 'string',
                                'description': 'What this question tests and why it aids learning.',
                            },
                        },
                        'required': ['question_type', 'question_style', 'focus'],
                        'additionalProperties': False,
                    },
                },
            },
            ['thought', 'questions'],
        )

    def _design_tools(self) -> list:
        return [
            self._fn_tool(
                'add_mcq_question',
                'Add a multiple-choice question (theory or practical) to the set.',
                {
                    'thought':        {'type': 'string', 'description': 'Why this MCQ is valuable and well-crafted for this concept.'},
                    'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                    'question':       {'type': 'string', 'description': 'The question stem.'},
                    'options':        {'type': 'array',  'items': {'type': 'string'}, 'description': 'Exactly 4 answer options, no A/B/C/D prefix.'},
                    'correct_index':  {'type': 'integer', 'description': '0-based index of the correct option.'},
                    'explanation':    {'type': 'string',  'description': 'Why the correct answer is right and the distractors are wrong.'},
                },
                ['thought', 'question_style', 'question', 'options', 'correct_index', 'explanation'],
            ),
            self._fn_tool(
                'add_open_question',
                'Add an open-ended question (theory or practical, optionally code) to the set.',
                {
                    'thought':        {'type': 'string',  'description': 'Why this open question is valuable for this concept.'},
                    'question_style': {'type': 'string',  'enum': ['theory', 'practical']},
                    'question':       {'type': 'string',  'description': 'The question text or code prompt.'},
                    'language':       {'type': 'string',  'enum': ['python', 'sql', 'none']},
                    'model_answer':   {'type': 'string',  'description': 'Ideal answer or reference solution.'},
                    'rubric':         {'type': 'string',  'description': 'Grading criteria — what a good answer must cover.'},
                    'test_code':      {'type': 'string',  'description': 'Test harness to auto-verify code answers. Empty string if N/A.'},
                },
                ['thought', 'question_style', 'question', 'language', 'model_answer', 'rubric', 'test_code'],
            ),
            self._fn_tool(
                'skip_type',
                'Skip a question category that is not feasible or valuable for this concept.',
                {
                    'thought':        {'type': 'string'},
                    'question_type':  {'type': 'string', 'enum': ['mcq', 'open']},
                    'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                    'reason':         {'type': 'string', 'description': 'Brief reason this type doesn\'t fit.'},
                },
                ['thought', 'question_type', 'question_style', 'reason'],
            ),
            self._fn_tool(
                'skip_question',
                'Skip a planned question that cannot be designed as intended.',
                {
                    'thought':      {'type': 'string'},
                    'plan_index':   {'type': 'integer', 'description': '1-based index of the planned question to skip.'},
                    'reason':       {'type': 'string', 'description': 'Why this question cannot be designed as planned.'},
                },
                ['thought', 'plan_index', 'reason'],
            ),
            self._fn_tool(
                'submit_question_set',
                'Finalise the question set after designing all planned questions.',
                {'thought': {'type': 'string', 'description': 'Summary of the designed set.'}},
                ['thought'],
            ),
        ]


question_engine = QuestionEngine(client)
print('✓ QuestionEngine ready')


✓ QuestionEngine ready


## Cell 9 — Answer Evaluation (`Grader`)
Three grading paths: **deterministic** for MCQ, **GPT-4o rubric** for open-ended prose, and **OpenAI Code Interpreter** execution for Python/SQL code answers.

In [9]:
class Grader:
    """Evaluates student answers for MCQ, open-ended, and Python/SQL code questions."""

    def __init__(self, openai_client):
        self.client = openai_client

    def grade(self, question: dict, student_answer: str) -> dict:
        """Route to the correct grader. Returns {score, correct, feedback}."""
        q_type   = question.get('question_type', 'open')
        language = question.get('language', 'none')
        if q_type == 'mcq':
            return self._grade_mcq(question, student_answer)
        if language in ('python', 'sql'):
            return self._grade_code(question, student_answer)
        return self._grade_open(question, student_answer)

    # ── MCQ (deterministic) ───────────────────────────────────────────────────

    def _grade_mcq(self, question: dict, student_answer: str) -> dict:
        try:
            chosen = int(student_answer)
        except (ValueError, TypeError):
            chosen = -1
        correct_idx = question.get('correct_index', -1)
        is_correct  = (chosen == correct_idx)
        explanation = question.get('explanation', '')
        return {
            'score':    1.0 if is_correct else 0.0,
            'correct':  is_correct,
            'feedback': (
                f'✅ Correct! {explanation}'
                if is_correct
                else f'❌ Incorrect. Right answer was option {correct_idx + 1}. {explanation}'
            )
        }

    # ── Open-ended (GPT-4o rubric) ────────────────────────────────────────────

    def _grade_open(self, question: dict, student_answer: str) -> dict:
        resp = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'Question: {question.get("question", "")}\n\n'
                f'Model answer: {question.get("model_answer", "")}\n\n'
                f'Rubric: {question.get("rubric", "")}\n\n'
                f'Student answer: {student_answer}\n\n'
                'Grade 0.0–1.0. Be encouraging but honest. '
                'Output ONLY valid JSON:\n'
                '{"score": 0.0, "correct": false, "feedback": "what was correct / missing / wrong"}'
            )
        )
        result = Ingester._parse_json(resp.output_text)
        result['correct'] = result.get('score', 0) >= 0.6
        return result

    # ── Code — Python / SQL (Code Interpreter) ────────────────────────────────

    def _grade_code(self, question: dict, student_answer: str) -> dict:
        language  = question.get('language', 'python')
        test_code = question.get('test_code', '')

        if language == 'sql':
            combined = (
                'import sqlite3\n'
                'conn = sqlite3.connect(":memory:")\n'
                'cur  = conn.cursor()\n\n'
                f'# --- Student SQL ---\n{student_answer}\n\n'
                f'# --- Tests ---\n{test_code}'
            )
        else:
            combined = f'# --- Student code ---\n{student_answer}\n\n# --- Tests ---\n{test_code}'

        try:
            resp = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'code_interpreter', 'container': {'type': 'auto'}}],
                input=(
                    f'Execute this {language.upper()} code and report whether all tests pass.\n\n'
                    f'```python\n{combined}\n```\n\n'
                    'After execution output ONLY valid JSON:\n'
                    '{"score": 1.0, "correct": true, "feedback": "...", "execution_output": "..."}'
                )
            )
            return Ingester._parse_json(resp.output_text)
        except Exception:
            return self._grade_open(question, student_answer)


grader = Grader(client)
print('✓ Grader ready')

✓ Grader ready


## Cell 10 — Study Session (`StudySession`)
Provides content recommendations for a session: weak topics first, then semantic similarity to recent sessions, then random fallback.

In [10]:
class StudySession:
    """Provides content recommendations for a study session."""

    def __init__(self, store: Store, vector_store: VectorStore):
        self.store        = store
        self.vector_store = vector_store

    def recommend_content(self, progress_tracker, n: int = 5) -> list:
        """
        Priority:
        1. Weak content (avg score < 0.6 over last 5 attempts)
        2. Semantically related to recent sessions
        3. Random sample of all content
        """
        # 1. Weak topics
        weak = progress_tracker.get_weak_content_ids()
        if weak:
            return weak[:n]

        # 2. Semantic similarity to recent sessions
        recent = progress_tracker.get_recent_sessions(n=3)
        if recent:
            titles = [
                c.title
                for s in recent
                if (c := self.store.get_content(s.content_id)) is not None
            ]
            if titles:
                found = self.vector_store.search(' '.join(titles), top_k=n)
                if found:
                    return found

        # 3. Random fallback
        all_ids = [c.id for c in self.store.all_content()]
        return random.sample(all_ids, min(n, len(all_ids)))


study_session = StudySession(store, vector_store)
print('✓ StudySession ready')

✓ StudySession ready


## Cell 11 — Progress Tracking (`ProgressTracker`)
Manages read/write access to `user_progress.json`. Computes weak topics (avg score < 60% over last 5 attempts), recent session history, study streak, and the full dashboard summary.

In [11]:
class ProgressTracker:
    """Tracks session history, weak topics, and study streaks."""

    WEAK_THRESHOLD = 0.6
    WEAK_WINDOW    = 5

    def __init__(self, store: Store):
        self.store = store

    def record(self, record: SessionRecord):
        self.store.sessions.append(record)
        self._recompute_weak()
        self.store._save_progress()

    def get_weak_content_ids(self) -> list:
        return list(self.store.weak_content_ids)

    def get_recent_sessions(self, n: int = 10) -> list:
        return self.store.sessions[-n:]

    def get_study_streak(self) -> int:
        """Count consecutive calendar days with at least one session."""
        if not self.store.sessions:
            return 0
        days   = sorted({s.date[:10] for s in self.store.sessions}, reverse=True)
        streak = 1
        for i in range(1, len(days)):
            delta = (date.fromisoformat(days[i - 1]) - date.fromisoformat(days[i])).days
            if delta == 1:
                streak += 1
            else:
                break
        return streak

    def summary(self) -> dict:
        recent = self.store.sessions[-10:]
        avg    = sum(s.score for s in recent) / len(recent) if recent else 0.0
        weak_titles = [
            c.title
            for cid in self.store.weak_content_ids
            if (c := self.store.get_content(cid)) is not None
        ]
        return {
            'total_content':    len(self.store.all_content()),
            'total_topics':     len(self.store.all_topics()),
            'total_sessions':   len(self.store.sessions),
            'avg_score_last10': avg,
            'weak_topics':      weak_titles,
            'streak_days':      self.get_study_streak(),
        }

    def _recompute_weak(self):
        scores_by: dict = {}
        for s in self.store.sessions:
            scores_by.setdefault(s.content_id, []).append(s.score)
        self.store.weak_content_ids = [
            cid
            for cid, scores in scores_by.items()
            if (sum(scores[-self.WEAK_WINDOW:]) / min(len(scores), self.WEAK_WINDOW))
               < self.WEAK_THRESHOLD
        ]


progress_tracker = ProgressTracker(store)
print('✓ ProgressTracker ready')

✓ ProgressTracker ready


## Cell 12 — Main Navigation UI
Button-based navigation inside a single `Output` container.
`clear_output(wait=True)` replaces the screen in-place on every navigation — no appended output.

**Run this cell to launch the AI Tutor.**

In [12]:
# ── Wire all classes together ──────────────────────────────────────────────────
ingester = Ingester(client, content_manager, vector_store, web_searcher,
                   suggest_similar=ENABLE_WEB_SUGGESTIONS,
                   show_reasoning=True)

# ── Helpers ────────────────────────────────────────────────────────────────────

def divider(char='─', width=56):
    print(char * width)

def header(title):
    print()
    divider('═')
    print(f'  {title}')
    divider('═')
    print()

def pause():
    input('  [Enter to continue] ')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MENU
# ══════════════════════════════════════════════════════════════════════════════

def main_menu():
    while True:
        header('🎓  AI Tutor')
        print('  [1] Manage content')
        print('  [2] Ingest new document')
        print('  [3] Study')
        print('  [4] Progress dashboard')
        print('  [0] Exit')
        print()
        c = input('  Choice: ').strip()
        if   c == '1': manage_screen()
        elif c == '2': ingest_screen()
        elif c == '3': study_screen()
        elif c == '4': progress_screen()
        elif c == '0': print('\n  Goodbye!\n'); break
        else:          print('  ⚠  Enter 0–4.')


# ══════════════════════════════════════════════════════════════════════════════
# MANAGE
# ══════════════════════════════════════════════════════════════════════════════

def manage_screen():
    def _normalize(text):
        return (text or '').strip().lower()

    def _topic_children_ids(tid):
        topic = store.get_topic(tid)
        if not topic:
            return []
        out_ids = []
        for cid in topic.children_topic_ids:
            out_ids.append(cid)
            out_ids.extend(_topic_children_ids(cid))
        return out_ids

    def _resolve_topic(ref):
        ref = (ref or '').strip()
        if not ref:
            return None
        topics = store.all_topics()
        exact_id = next((t for t in topics if t.id == ref), None)
        if exact_id: return exact_id
        prefix_id = next((t for t in topics if t.id.startswith(ref)), None)
        if prefix_id: return prefix_id
        key = _normalize(ref)
        exact_name = [t for t in topics if _normalize(t.name) == key]
        if len(exact_name) == 1: return exact_name[0]
        by_contains = [t for t in topics if key in _normalize(t.name)]
        return by_contains[0] if len(by_contains) == 1 else None

    def _resolve_content(ref):
        ref = (ref or '').strip()
        if not ref:
            return None
        content = store.all_content()
        exact_id = next((c for c in content if c.id == ref), None)
        if exact_id: return exact_id
        prefix_id = next((c for c in content if c.id.startswith(ref)), None)
        if prefix_id: return prefix_id
        key = _normalize(ref)
        exact_title = [c for c in content if _normalize(c.title) == key]
        if len(exact_title) == 1: return exact_title[0]
        by_contains = [c for c in content if key in _normalize(c.title)]
        return by_contains[0] if len(by_contains) == 1 else None

    def _help_text():
        return (
            'Commands:\n'
            '  help\n'
            '  show tree\n'
            '  create topic <name> [under <parent>]\n'
            '  rename topic <ref> -> <new_name>\n'
            '  rename content <ref> -> <new_title>\n'
            '  move topic <ref> to <parent_ref>\n'
            '  move content <ref> to <topic_ref>\n'
            '  delete topic <ref>\n'
            '  delete content <ref>'
        )

    def _execute_action(action):
        action_type = (action.get('action') or '').strip().lower()
        args = action.get('args') or {}

        if action_type in ('help', 'unknown'):
            return _help_text()
        if action_type == 'show_tree':
            return content_manager.list_topics()
        if action_type == 'create_topic':
            name = (args.get('name') or '').strip()
            parent_ref = (args.get('parent_ref') or '').strip()
            if not name: return '⚠ Topic name required.'
            parent = _resolve_topic(parent_ref) if parent_ref else store.root_topic()
            if not parent: return f'⚠ Parent not found: {parent_ref}'
            t = content_manager.create_topic(name, parent.id)
            return f'✓ Created topic "{t.name}" under "{parent.name}".'
        if action_type == 'rename_topic':
            topic = _resolve_topic(args.get('topic_ref', ''))
            new_name = (args.get('new_name') or '').strip()
            if not topic: return f'⚠ Topic not found.'
            old = topic.name; topic.name = new_name; store._save_tree()
            return f'✓ Renamed "{old}" -> "{new_name}".'
        if action_type == 'rename_content':
            c = _resolve_content(args.get('content_ref', ''))
            new_title = (args.get('new_title') or '').strip()
            if not c: return '⚠ Content not found.'
            old = c.title; c.title = new_title; store._save_tree()
            return f'✓ Renamed "{old}" -> "{new_title}".'
        if action_type == 'move_content':
            c = _resolve_content(args.get('content_ref', ''))
            t = _resolve_topic(args.get('topic_ref', ''))
            if not c: return '⚠ Content not found.'
            if not t: return '⚠ Topic not found.'
            content_manager.move_content(c.id, t.id)
            return f'✓ Moved "{c.title}" to "{t.name}".'
        if action_type == 'move_topic':
            topic = _resolve_topic(args.get('topic_ref', ''))
            new_parent = _resolve_topic(args.get('new_parent_ref', ''))
            if not topic: return '⚠ Topic not found.'
            if not new_parent: return '⚠ Destination not found.'
            if topic.id == store.root_topic_id: return '⚠ Root cannot be moved.'
            if new_parent.id in _topic_children_ids(topic.id): return '⚠ Cannot move under descendant.'
            old_parent = store.get_topic(topic.parent_id)
            if old_parent and topic.id in old_parent.children_topic_ids:
                old_parent.children_topic_ids.remove(topic.id)
            topic.parent_id = new_parent.id
            if topic.id not in new_parent.children_topic_ids:
                new_parent.children_topic_ids.append(topic.id)
            store._save_tree()
            return f'✓ Moved "{topic.name}" under "{new_parent.name}".'
        if action_type == 'delete_content':
            c = _resolve_content(args.get('content_ref', ''))
            if not c: return '⚠ Content not found.'
            file_id = content_manager.delete_content(c.id)
            vector_store.delete_content(file_id)
            return f'✓ Deleted "{c.title}".'
        if action_type == 'delete_topic':
            topic = _resolve_topic(args.get('topic_ref', ''))
            if not topic: return '⚠ Topic not found.'
            if topic.id == store.root_topic_id: return '⚠ Root cannot be deleted.'
            parent = store.get_topic(topic.parent_id)
            if not parent: return '⚠ Parent missing.'
            for child_id in list(topic.children_topic_ids):
                child = store.get_topic(child_id)
                if child:
                    child.parent_id = parent.id
                    if child.id not in parent.children_topic_ids:
                        parent.children_topic_ids.append(child.id)
            for cid in list(topic.content_ids):
                c = store.get_content(cid)
                if c:
                    c.topic_id = parent.id
                    if cid not in parent.content_ids:
                        parent.content_ids.append(cid)
            if topic.id in parent.children_topic_ids:
                parent.children_topic_ids.remove(topic.id)
            if topic.id in store.topics:
                del store.topics[topic.id]
            store._save_tree()
            return f'✓ Deleted topic "{topic.name}", children reparented to "{parent.name}".'
        return '⚠ Unsupported action.'

    def _run_command(cmd):
        raw = (cmd or '').strip()
        if not raw: return '⚠ Enter a command. Type: help'
        if raw.lower() in ('help', '?'): return _help_text()

        topic_state   = [{'id': t.id, 'name': t.name, 'parent_id': t.parent_id} for t in store.all_topics()]
        content_state = [{'id': c.id, 'title': c.title, 'topic_id': c.topic_id} for c in store.all_content()]
        schema_hint = {
            'action': 'one_of(help,show_tree,create_topic,rename_topic,rename_content,move_topic,move_content,delete_topic,delete_content,unknown)',
            'args': {'name': 'str', 'parent_ref': 'str', 'topic_ref': 'str', 'new_parent_ref': 'str',
                     'content_ref': 'str', 'new_name': 'str', 'new_title': 'str'},
            'reason': 'str'
        }
        prompt = (
            f'You are a content-management command planner.\n'
            f'Output ONLY valid JSON: {json.dumps(schema_hint)}\n\n'
            f'Topics: {json.dumps(topic_state)}\n'
            f'Content: {json.dumps(content_state)}\n'
            f'Command: {raw}'
        )
        try:
            resp = client.responses.create(model='gpt-4o', input=prompt, temperature=0)
            plan = json.loads(resp.output_text.strip())
        except Exception as e:
            return f'⚠ Agent error: {e}'
        return _execute_action(plan)

    while True:
        header('📁  Manage Content')
        print(content_manager.list_topics())
        print()
        print('  Type a command (or "help", "0" to go back):')
        print()
        cmd = input('  > ').strip()
        if cmd == '0':
            break
        result = _run_command(cmd)
        print()
        print(result)
        print()
        pause()


# ══════════════════════════════════════════════════════════════════════════════
# INGEST
# ══════════════════════════════════════════════════════════════════════════════

def ingest_screen():
    if IS_COLAB:
        _ingest_colab()
    else:
        _ingest_local()


def _ingest_colab():
    header('⬆️   Ingest Document')
    print('  A file picker will open — select a PDF or image from your computer.')
    print()
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as exc:
        print(f'  ❌  Upload failed: {exc}')
        return
    if not uploaded:
        print('  ⚠  No file selected.')
        return
    for filename, data in uploaded.items():
        ext = Path(filename).suffix.lower()
        if ext not in Ingester.SUPPORTED_EXTS:
            print(f'  ⚠  Skipping "{filename}" — unsupported type "{ext}".')
            continue
        print(f'\n  Ingesting "{filename}"…\n')
        try:
            ingester.ingest(data, filename, log_fn=print)
        except Exception as exc:
            print(f'\n  ❌  Ingest failed: {exc}')
    print()
    pause()


def _ingest_local():
    header('⬆️   Ingest Document')

    # Try native file picker first
    path_str = None
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.call('wm', 'attributes', '.', '-topmost', True)
        exts = ' '.join(f'*{e}' for e in Ingester.SUPPORTED_EXTS)
        path_str = filedialog.askopenfilename(
            title='Select a PDF or image file',
            filetypes=[('Supported files', exts), ('All files', '*.*')],
        )
        root.destroy()
        if path_str:
            print(f'  Selected: {path_str}')
        else:
            print('  No file selected.')
            return
    except Exception:
        # tkinter not available — fall back to manual path input
        print('  Enter the full path to a PDF or image file (or 0 to cancel):')
        print()
        path_str = input('  File path: ').strip()
        if path_str == '0':
            return

    path = Path(path_str).expanduser()
    if not path.exists() or not path.is_file():
        print(f'  ⚠  File not found: {path}')
        return
    ext = path.suffix.lower()
    if ext not in Ingester.SUPPORTED_EXTS:
        print(f'  ⚠  Unsupported type "{ext}". Supported: {Ingester.SUPPORTED_EXTS}')
        return
    print()
    try:
        data = path.read_bytes()
        ingester.ingest(data, path.name, log_fn=print)
    except Exception as exc:
        print(f'\n  ❌  Ingest failed: {exc}')
    print()
    pause()


# ══════════════════════════════════════════════════════════════════════════════
# STUDY
# ══════════════════════════════════════════════════════════════════════════════

def study_screen():
    while True:
        header('📖  Study')
        print('  [1] Recommended  — system picks weak / related topics')
        print('  [2] Manual       — choose concepts yourself')
        print('  [0] Back')
        print()
        c = input('  Choice: ').strip()
        if c == '0':
            break
        elif c == '1':
            queue = study_session.recommend_content(progress_tracker)
            if not queue:
                print('  ⚠  No content available. Ingest documents first.')
                continue
            _study_run(queue)
            break
        elif c == '2':
            queue = _study_pick_manual()
            if queue:
                _study_run(queue)
            break
        else:
            print('  ⚠  Enter 0, 1, or 2.')


def _study_pick_manual():
    contents = store.all_content()
    if not contents:
        print('  ⚠  No content available. Ingest documents first.')
        return []
    print()
    print('  Available concepts:')
    for i, c in enumerate(contents, 1):
        print(f'    [{i}] {c.title}')
    print()
    raw = input('  Enter numbers separated by commas (e.g. 1,3): ').strip()
    selected = []
    for part in raw.split(','):
        try:
            idx = int(part.strip()) - 1
            if 0 <= idx < len(contents):
                selected.append(contents[idx].id)
        except ValueError:
            pass
    if not selected:
        print('  ⚠  No valid selection.')
    return selected


def _study_run(queue):
    all_scores = []

    for c_idx, content_id in enumerate(queue):
        content_obj = store.get_content(content_id)
        if not content_obj:
            continue

        header(f'📐  Designing Questions')
        print(f'  Concept  : {content_obj.title}')
        print(f'  Progress : {c_idx + 1} of {len(queue)}')
        print()

        try:
            questions = question_engine.design_question_set(content_obj, log_fn=print)
        except Exception as exc:
            print(f'\n  ❌  Design error: {exc}')
            c = input('  [1] Skip  [0] Quit: ').strip()
            if c == '0': break
            continue

        if not questions:
            print('  ⚠  No questions designed — skipping.')
            continue

        print(f'\n  ✓ {len(questions)} question(s) ready.\n')

        q_idx    = 0
        fu_count = 0
        cur_q    = None

        while q_idx < len(questions):
            if cur_q is None:
                cur_q = questions[q_idx]

            q_type   = cur_q.get('question_type', 'open')
            q_style  = cur_q.get('question_style', 'theory')
            q_text   = cur_q.get('question', '')
            language = cur_q.get('language', 'none')
            label    = f'Q{q_idx + 1}/{len(questions)}' + (f'  (follow-up {fu_count})' if fu_count else '')

            print()
            divider('═')
            print(f'  📖  {label}')
            print(f'  Concept : {content_obj.title}')
            print(f'  Format  : {q_type.upper()}  |  Style: {q_style.title()}')
            divider()
            print()
            for line in q_text.splitlines():
                print(f'  {line}')
            print()

            if q_type == 'mcq':
                options = cur_q.get('options', [])
                for i, opt in enumerate(options):
                    print(f'  {chr(65 + i)}. {opt}')
                print()
                raw_ans = input('  Your answer (A/B/C/D): ').strip().upper()
                student_answer = str(ord(raw_ans[0]) - ord('A')) if raw_ans and raw_ans[0] in 'ABCD' else '-1'
            else:
                hint = f'Write your {language.upper()} code' if language in ('python', 'sql') else 'Type your answer'
                print(f'  {hint} (blank line to submit):')
                lines = []
                while True:
                    line = input()
                    if line == '':
                        break
                    lines.append(line)
                student_answer = '\n'.join(lines)

            print()
            print('  ⏳  Grading…')
            result  = grader.grade(cur_q, student_answer)
            score   = result.get('score', 0.0)
            verdict = '✅  Correct!' if score >= 0.6 else '❌  Needs review'
            all_scores.append(score)

            progress_tracker.record(SessionRecord(
                date=datetime.utcnow().isoformat(),
                content_id=content_obj.id,
                question_type=q_type,
                correct=result.get('correct', score >= 0.6),
                score=score,
            ))

            print()
            divider()
            print(f'  Score : {score:.0%}   {verdict}')
            divider()
            print()
            for line in result.get('feedback', '').splitlines():
                print(f'  {line}')
            print()
            print(f'  Session: {len(all_scores)} answered  |  avg {sum(all_scores)/len(all_scores):.0%}')
            print()

            fu = None
            if q_type == 'open' and fu_count < QuestionEngine.MAX_FOLLOWUPS and score < 0.9:
                print('  ⏳  Considering follow-up…')
                fu = question_engine.generate_followup(content_obj, cur_q, student_answer, score)

            print('  [1] Next question')
            print('  [0] Quit session')
            print()
            nav = input('  Choice: ').strip()
            if nav == '0':
                _study_complete(all_scores)
                return

            if fu:
                fu_count += 1
                cur_q = fu
            else:
                q_idx   += 1
                fu_count = 0
                cur_q    = None

    _study_complete(all_scores)


def _study_complete(all_scores):
    avg = sum(all_scores) / len(all_scores) if all_scores else 0.0
    header('🎉  Session Complete!')
    print(f'  Questions answered : {len(all_scores)}')
    print(f'  Average score      : {avg:.0%}')
    print()
    if avg >= 0.8:
        print('  Great work! Keep it up.')
    elif avg >= 0.6:
        print('  Good effort. Review flagged topics and try again.')
    else:
        print('  Several weak areas — recommended session will focus on them.')
    print()


# ══════════════════════════════════════════════════════════════════════════════
# PROGRESS
# ══════════════════════════════════════════════════════════════════════════════

def progress_screen():
    s      = progress_tracker.summary()
    recent = progress_tracker.get_recent_sessions(10)

    header('📊  Progress Dashboard')
    print(f'  Content items  : {s["total_content"]}')
    print(f'  Topics         : {s["total_topics"]}')
    print(f'  Total sessions : {s["total_sessions"]}')
    print(f'  Avg (last 10)  : {s["avg_score_last10"]:.0%}')
    print(f'  Study streak   : {s["streak_days"]} day(s)')
    divider()
    weak = s['weak_topics']
    if weak:
        print()
        print('  ⚠  Weak topics (avg < 60% over last 5 attempts):')
        for w in weak:
            print(f'     • {w}')
    else:
        print()
        print('  ✅  No weak topics — great work!')
    divider()
    print()
    print('  Recent sessions (newest first):')
    print()
    if not recent:
        print('    (no sessions yet)')
    else:
        print(f'    {"Date":<12} {"Score":>6}  {"Type":<10}  Concept')
        print(f'    {"─"*10:<12} {"─"*5:>6}  {"─"*8:<10}  {"─"*20}')
        for r in reversed(recent):
            c   = store.get_content(r.content_id)
            ttl = (c.title[:28] + '…') if c and len(c.title) > 28 else (c.title if c else r.content_id[:8])
            mark = '✅' if r.correct else '❌'
            print(f'    {r.date[:10]:<12} {r.score:>5.0%}  {r.question_type:<10}  {mark} {ttl}')
    print()
    pause()


# ══════════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════════

main_menu()



════════════════════════════════════════════════════════
  🎓  AI Tutor
════════════════════════════════════════════════════════

  [1] Manage content
  [2] Ingest new document
  [3] Study
  [4] Progress dashboard
  [0] Exit


════════════════════════════════════════════════════════
  📁  Manage Content
════════════════════════════════════════════════════════

📁 Root
  📄 Matrix Factorisation in Collaborative Filtering
  📄 Latent Factors in Matrix Factorisation
  📄 Gradient Descent for Optimisation
  📄 Batch vs. Stochastic Gradient Descent
  📄 Overfitting and Regularisation in Matrix Factorisation
  📄 User and Item Biases in Matrix Factorisation
  📄 Alternating Least Squares in Matrix Factorisation
  📄 Weighted Matrix Factorisation for Implicit Data
  📄 Top N Recommendation Strategies in Matrix Factorisation
  📁 Machine Learning
    📁 Recommender Systems
      📄 Non-personalised vs. Personalised Recommendation
      📄 Personalized vs. Non-personalized Recommendations
      📄 User Profile

/var/folders/8b/s8wf1mns4pd7rjv696_thjsr0000gn/T/ipykernel_51436/1255509192.py:449: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date=datetime.utcnow().isoformat(),



════════════════════════════════════════════════════════
  📖  Q2/6
  Concept : Alternating Least Squares (ALS) for Matrix Factorization
  Format  : OPEN  |  Style: Theory
────────────────────────────────────────────────────────

  Describe the step-by-step process of the Alternating Least Squares (ALS) method for matrix factorization, focusing on its iterative nature and how it solves for latent matrices.

  Type your answer (blank line to submit):

  ⏳  Grading…

────────────────────────────────────────────────────────
  Score : 0%   ❌  Needs review
────────────────────────────────────────────────────────

  Understanding ALS requires outlining the iterative process where one latent matrix is fixed while solving for the other, aiming for minimal error in matrix approximation. The response lacked these elements and did not explain the crucial steps of the algorithm.

  Session: 2 answered  |  avg 0%

  ⏳  Considering follow-up…
  [1] Next question
  [0] Quit session


════════════════

KeyboardInterrupt: Interrupted by user